# F1 Enterprise Analytics: Master Data Warehouse
**Techniques:** Data Warehousing (14 Relational Tables), Random Forest Ensemble, K-Means Clustering, PCA Dimensionality Reduction
**Author:** Data Mining Project Submission

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import classification_report, accuracy_score, silhouette_score

plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={
    "axes.facecolor": "#0a0f18",
    "figure.facecolor": "#0a0f18",
    "grid.color": "#1a2333",
    "text.color": "#f0f4f8", 
    "axes.labelcolor": "#00e5ff",
    "xtick.color": "#a0aec0",
    "ytick.color": "#a0aec0",
    "font.family": "monospace"
})
palette = ["#00e5ff", "#ff0055", "#ffaa00", "#b200ff", "#00ff88"]

### 1. Data Ingestion Protocol

In [ ]:
print("Initializing Data Ingestion Protocol...")

circuits = pd.read_csv('circuits.csv')
constructors = pd.read_csv('constructors.csv')
drivers = pd.read_csv('drivers.csv')
races = pd.read_csv('races.csv')
seasons = pd.read_csv('seasons.csv')
status = pd.read_csv('status.csv')

results = pd.read_csv('results.csv')
qualifying = pd.read_csv('qualifying.csv')
sprint_results = pd.read_csv('sprint_results.csv')

lap_times = pd.read_csv('lap_times.csv')
pit_stops = pd.read_csv('pit_stops.csv')

driver_standings = pd.read_csv('driver_standings.csv')
constructor_standings = pd.read_csv('constructor_standings.csv')
constructor_results = pd.read_csv('constructor_results.csv')

print("All 14 databases successfully loaded into memory.")

### 2. Advanced Feature Engineering

In [ ]:
print("Engineering Advanced Telemetry Features...")

lap_consistency = lap_times.groupby(['raceId', 'driverId']).agg(
    avg_lap_ms=('milliseconds', 'mean'),
    lap_std_dev=('milliseconds', 'std')
).reset_index()

pit_efficiency = pit_stops.groupby(['raceId', 'driverId']).agg(
    total_pit_stops=('stop', 'max'),
    avg_pit_duration=('milliseconds', 'mean')
).reset_index()

sprint_impact = sprint_results[['raceId', 'driverId', 'points']].rename(
    columns={'points': 'sprint_points'}
)

print("Pre-aggregation complete.")

### 3. Master Data Warehouse Merge

In [ ]:
print("Executing Relational Master Merge...")

df = results.copy()
df.replace('\\N', np.nan, inplace=True)

df = df.merge(races[['raceId', 'year', 'circuitId']], on='raceId', how='left')
df = df.merge(circuits[['circuitId', 'name', 'location']], on='circuitId', how='left', suffixes=('', '_circuit'))
df = df.merge(drivers[['driverId', 'dob', 'driverRef']], on='driverId', how='left')
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left', suffixes=('', '_team'))
df = df.merge(status[['statusId', 'status']], on='statusId', how='left')

df = df.merge(lap_consistency, on=['raceId', 'driverId'], how='left')
df = df.merge(pit_efficiency, on=['raceId', 'driverId'], how='left')
df = df.merge(sprint_impact, on=['raceId', 'driverId'], how='left')
df['sprint_points'] = df['sprint_points'].fillna(0)

df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')

df['Podium'] = df['positionOrder'].apply(lambda x: 1 if x <= 3 else 0)
df_clean = df.dropna(subset=['lap_std_dev', 'avg_pit_duration', 'grid']).copy()

print(f"Master Dataset Assembled. Shape: {df_clean.shape}")

### 4. Enterprise Visualization Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('F1 NEURAL VAULT: SYSTEM ANALYTICS', fontsize=24, color='#00e5ff', fontweight='bold', y=0.98)

sns.violinplot(ax=axes[0, 0], data=df_clean, x='Podium', y='lap_std_dev', palette=['#ff0055', '#00e5ff'], inner="quartile")
axes[0, 0].set_title('DRIVER CONSISTENCY (LAP VARIANCE) BY PODIUM OUTCOME', color='#ffaa00')
axes[0, 0].set_ylim(0, 150000)
axes[0, 0].set_ylabel('Lap Time Standard Deviation (ms)')

top_teams = df_clean['name_team'].value_counts().head(5).index
team_data = df_clean[df_clean['name_team'].isin(top_teams)]
sns.boxplot(ax=axes[0, 1], data=team_data, x='name_team', y='avg_pit_duration', palette=palette)
axes[0, 1].set_title('PIT WALL EFFICIENCY: CONSTRUCTOR ANALYSIS', color='#ffaa00')
axes[0, 1].set_ylim(20000, 40000)

sns.scatterplot(ax=axes[1, 0], data=df_clean, x='grid', y='points', hue='Podium', palette=['#334155', '#00ff88'], alpha=0.6)
axes[1, 0].set_title('STARTING GRID VS CHAMPIONSHIP POINTS YIELD', color='#ffaa00')

features_to_corr = ['grid', 'lap_std_dev', 'total_pit_stops', 'avg_pit_duration', 'sprint_points', 'positionOrder']
corr_matrix = df_clean[features_to_corr].corr()
sns.heatmap(corr_matrix, ax=axes[1, 1], annot=True, cmap='mako', linewidths=1, linecolor='#0a0f18', fmt='.2f')
axes[1, 1].set_title('TELEMETRY CORRELATION MATRIX', color='#ffaa00')

plt.tight_layout(pad=3.0)
plt.show()

### 5. Machine Learning Models: Random Forest & PCA Clustering

In [ ]:
print("\n--- INITIATING PREDICTIVE MODELS ---")

features = ['grid', 'lap_std_dev', 'avg_pit_duration', 'total_pit_stops', 'sprint_points', 'constructorId']
X = df_clean[features]
y = df_clean['Podium']

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("\n[+] RANDOM FOREST: PODIUM PREDICTION RESULTS")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

importance = pd.DataFrame({'Feature': features, 'Importance': rf_model.feature_importances_})
importance = importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=importance, x='Importance', y='Feature', palette='viridis')
plt.title("NEURAL NETWORK FEATURE IMPORTANCE", color='#00e5ff')
plt.show()

print("\n[+] K-MEANS CLUSTERING & PCA DIMENSIONALITY REDUCTION")

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_pca)

print(f"Silhouette Score (Cluster Validity): {silhouette_score(X_pca, clusters):.3f}")

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='plasma', alpha=0.5, edgecolor='#0a0f18')
plt.title("PCA VISUALIZATION: 4-DIMENSIONAL DRIVER PROFILES", color='#00e5ff', fontsize=14)
plt.xlabel("Principal Component 1 (Pace & Grid Influence)")
plt.ylabel("Principal Component 2 (Pit Strategy Influence)")
plt.colorbar(scatter, label='Strategy Cluster')
plt.show()